# 03. Baseline models and error diagnostics

In [1]:
from pathlib import Path
import re, json, hashlib, time, gc, warnings, itertools, platform
import numpy as np
import pandas as pd
import yaml, joblib
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, matthews_corrcoef, balanced_accuracy_score,
    brier_score_loss, log_loss, confusion_matrix)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                 if (path / "dataset").is_dir() and (path / "notebooks").is_dir())
OUTPUT_ROOT = REPO_ROOT / "artifacts"  # generated tables, models, predictions and figures
(OUTPUT_ROOT / "Configs").mkdir(parents=True, exist_ok=True)
SHARED_DATASET_ROOT = REPO_ROOT / "dataset"
for folder in ["Processed", "Interim", "Features", "Splits"]:
    (SHARED_DATASET_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_ID = "racf_20261011"
TABLES = OUTPUT_ROOT / "Tables" / RUN_ID
MODEL_ROOT = OUTPUT_ROOT / "Models" / RUN_ID / "baselines"
PRED_ROOT = OUTPUT_ROOT / "Predictions" / RUN_ID / "baselines"
for folder in [TABLES, MODEL_ROOT, PRED_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.precision", 4)
protocol = yaml.safe_load((OUTPUT_ROOT / "Configs/experiment.yaml").read_text())
assert not protocol["test_opened_for_model_selection"]
SEEDS = protocol["seeds"]
N_JOBS = 4
thread_limiter = threadpool_limits(limits=N_JOBS)


In [2]:
def scientific_table(frame: pd.DataFrame, name: str, caption: str) -> None:
    """Export and validate all results; display the selected narrative tables."""
    frame.to_csv(TABLES / f"{name}.csv", index=False)
    assert not frame.isna().any().any(), f"Explain undefined statistics in {name}"
    # Category-based curation applies to all models, irrespective of their results.
    if name.startswith(("nb03_trials_", "nb03_seeds_", "nb03_regime_",
                        "nb03_site_errors_", "nb03_reactor_errors_")):
        return
    display(Markdown(f"**{caption}**"))
    display(frame.style.format(scientific_number).hide(axis="index"))


def scientific_number(value, decimals=4):
    """Typography only: preserve raw numeric values and Python arithmetic."""
    if pd.isna(value):
        raise ValueError("An undefined statistic requires an explicit scientific explanation")
    if isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_)):
        return str(value).replace("-", "−")
    if isinstance(value, (float, np.floating)):
        if np.isinf(value):
            return "∞" if value>0 else "−∞"
        if value == 0:
            return f"{0:.{decimals}f}"
        if abs(value) < 10**(-decimals):
            return f"{value:.2e}".replace("-", "−")
        return f"{value:.{decimals}f}".replace("-", "−")
    return re.sub(r"(?<![A-Za-z0-9])-(?=\d)|(?<=[eE])-(?=\d)", "−", str(value))


## 1. Fixed data and evaluation boundaries

In [3]:
FEATURES_ROOT = SHARED_DATASET_ROOT / "Features"
SPLITS_ROOT = SHARED_DATASET_ROOT / "Splits"
features = pd.read_parquet(FEATURES_ROOT / "features_raw.parquet")
metadata = pd.read_parquet(FEATURES_ROOT / "observation_metadata.parquet")
schema = json.loads((FEATURES_ROOT / "feature_schema.json").read_text())
assert features.columns.tolist() == schema["feature_names"]
splits = {name:{role:np.load(SPLITS_ROOT / f"{name}_{role}.npy") for role in ["train","evaluation"]}
          for name in ["forward_2020","forward_2021","forward_2022","development","site_transfer"]}
assert len(features) == len(metadata) and len(features.columns) == 51
assert metadata.iloc[splits["development"]["train"]].date.max().year == 2022
assert metadata.iloc[splits["development"]["evaluation"]].date.max().year == 2023
transfer_sites = set(metadata.iloc[splits["site_transfer"]["evaluation"]].plant_id)
# The full predeclared assignment also excludes held sites absent from 2023 evaluation.
site_assignment = pd.read_csv(SPLITS_ROOT / "site_holdout_assignment.csv")
scientific_table(pd.DataFrame([{"Partition":name, "Training n":len(part["train"]),
    "Evaluation n":len(part["evaluation"]), "Positive days":int(metadata.iloc[part["evaluation"]].target.sum()),
    "Evaluation sites":metadata.iloc[part["evaluation"]].plant_id.nunique()}
    for name, part in splits.items()]), "nb03_evaluation_cohorts", "Common evaluation cohorts; positive days are not independent observations")


**Common evaluation cohorts; positive days are not independent observations**

Partition,Training n,Evaluation n,Positive days,Evaluation sites
forward_2020,28287,20555,46,54
forward_2021,57744,19351,36,54
forward_2022,87425,19020,35,51
development,116904,20333,42,50
site_transfer,80198,5634,15,15


In [4]:
# Inspect the predeclared site flag by its fixed schema, without consulting outcomes.
assert site_assignment.columns.tolist() == ["plant_id", "held_out"]
held_sites = set(site_assignment.loc[site_assignment.held_out, "plant_id"])
assert transfer_sites <= held_sites
forward_names = ["forward_2020", "forward_2021", "forward_2022"]
search_splits = {"temporal":{name:splits[name] for name in forward_names},
    "transfer":{name:{role:idx[~metadata.iloc[idx].plant_id.isin(held_sites).to_numpy()]
                      for role,idx in splits[name].items()} for name in forward_names}}
for scope, partitions in search_splits.items():
    for fold, part in partitions.items():
        train_meta, eval_meta = metadata.iloc[part["train"]], metadata.iloc[part["evaluation"]]
        assert (eval_meta.date.min() - train_meta.date.max()).days > 30
        assert set(train_meta.episode_id).isdisjoint(set(eval_meta.episode_id))
        if scope == "transfer":
            assert set(train_meta.plant_id).isdisjoint(held_sites)
            assert set(eval_meta.plant_id).isdisjoint(held_sites)


## 2. Models and prespecified configurations

In [5]:
GRIDS = {
    "Logistic Regression":[{"C":0.1,"weight":"balanced"},{"C":1.0,"weight":"balanced"},{"C":1.0,"weight":"none"}],
    "Random Forest":[{"leaf":1,"weight":"balanced_subsample"},{"leaf":5,"weight":"balanced_subsample"},{"leaf":2,"weight":"none"}],
    "CatBoost":[{"depth":4,"weight":"none"},{"depth":4,"weight":"sqrt"},{"depth":6,"weight":"sqrt"}],
    "LightGBM":[{"leaves":15,"weight":"none"},{"leaves":15,"weight":"sqrt"},{"leaves":31,"weight":"sqrt"}]}
FAMILIES = list(GRIDS)
model_slugs = {"Majority":"majority", "Power-drop rule":"power_drop", "Recent-level deviation":"level_deviation",
               "Logistic Regression":"logistic", "Random Forest":"random_forest", "CatBoost":"catboost","LightGBM":"lightgbm"}
BASE_SETTINGS = {"Logistic Regression":{"solver":"lbfgs","max_iter":1500},
    "Random Forest":{"n_estimators":400,"max_features":0.7,"n_jobs":N_JOBS},
    "CatBoost":{"iterations":800,"learning_rate":0.04,"l2_leaf_reg":10,"thread_count":N_JOBS},
    "LightGBM":{"n_estimators":800,"learning_rate":0.04,"min_child_samples":20,"reg_lambda":5,"n_jobs":N_JOBS}}
search_spec = {"run_id":RUN_ID,"grids":GRIDS,"fixed":BASE_SETTINGS,"folds":forward_names,
    "selection":"mean forward-fold AP; predeclared configuration order for ties","search_seed":42,"repeat_seeds":SEEDS,
    "site_selection":"separate search restricted to training sites","calibration":"development sigmoid from OOF; final 2024 calibration deferred",
    "rule_drop_pp":20,"rule_recent_level_pp":20,"rule_recent_window_days":6}
(OUTPUT_ROOT / "Configs/baseline_search.json").write_text(json.dumps(search_spec,indent=2))
grid_rows = [{"Model":family,"Configuration":k+1,"Parameters":json.dumps(config),
    "Fixed settings":json.dumps(BASE_SETTINGS[family])} for family,grid in GRIDS.items() for k,config in enumerate(grid)]
for family in FAMILIES:
    scientific_table(pd.DataFrame(grid_rows).query("Model == @family").drop(columns="Model"),
        "nb03_search_plan_"+model_slugs[family], f"{family}: configurations specified before training")


def make_model(family, config, seed, y_train):
    settings = BASE_SETTINGS[family].copy()
    if family == "Logistic Regression":
        clf = LogisticRegression(**settings, C=config["C"], class_weight=None if config["weight"] == "none" else config["weight"], random_state=seed)
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),
                         ("scaler",StandardScaler()),("classifier",clf)])
    if family == "Random Forest":
        clf = RandomForestClassifier(**settings, min_samples_leaf=config["leaf"],
            class_weight=None if config["weight"] == "none" else config["weight"], random_state=seed)
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("classifier",clf)])
    if family == "CatBoost":
        return CatBoostClassifier(**settings, depth=config["depth"], random_seed=seed, loss_function="Logloss",
            auto_class_weights="SqrtBalanced" if config["weight"] == "sqrt" else None,
            verbose=False, allow_writing_files=False)
    if family == "LightGBM":
        ratio = (len(y_train)-y_train.sum()) / max(y_train.sum(),1)
        return LGBMClassifier(**settings,num_leaves=config["leaves"],random_state=seed,verbosity=-1,
            scale_pos_weight=np.sqrt(ratio) if config["weight"] == "sqrt" else 1.0,
            deterministic=True,force_col_wise=True)
    raise ValueError(family)


def make_predictions(indices, raw, family, seed, fold, scope, config_id):
    keep = ["observation_id","reactor_id","plant_id","date","episode_id","event_episode_id",
            "event_id","target","control_kind","label_source","scram_type"]
    out = metadata.iloc[indices][keep].copy()
    out["row_index"], out["raw_score"] = indices, np.asarray(raw,float)
    out["model"],out["seed"],out["fold"],out["scope"],out["configuration"] = family,seed,fold,scope,config_id
    assert out.observation_id.is_unique and np.isfinite(out.raw_score).all()
    return out.reset_index(drop=True)


search_rows, prediction_frames, resource_rows, chosen = [], [], [], {}


**Logistic Regression: configurations specified before training**

Configuration,Parameters,Fixed settings
1,"{""C"": 0.1, ""weight"": ""balanced""}","{""solver"": ""lbfgs"", ""max_iter"": 1500}"
2,"{""C"": 1.0, ""weight"": ""balanced""}","{""solver"": ""lbfgs"", ""max_iter"": 1500}"
3,"{""C"": 1.0, ""weight"": ""none""}","{""solver"": ""lbfgs"", ""max_iter"": 1500}"


**Random Forest: configurations specified before training**

Configuration,Parameters,Fixed settings
1,"{""leaf"": 1, ""weight"": ""balanced_subsample""}","{""n_estimators"": 400, ""max_features"": 0.7, ""n_jobs"": 4}"
2,"{""leaf"": 5, ""weight"": ""balanced_subsample""}","{""n_estimators"": 400, ""max_features"": 0.7, ""n_jobs"": 4}"
3,"{""leaf"": 2, ""weight"": ""none""}","{""n_estimators"": 400, ""max_features"": 0.7, ""n_jobs"": 4}"


**CatBoost: configurations specified before training**

Configuration,Parameters,Fixed settings
1,"{""depth"": 4, ""weight"": ""none""}","{""iterations"": 800, ""learning_rate"": 0.04, ""l2_leaf_reg"": 10, ""thread_count"": 4}"
2,"{""depth"": 4, ""weight"": ""sqrt""}","{""iterations"": 800, ""learning_rate"": 0.04, ""l2_leaf_reg"": 10, ""thread_count"": 4}"
3,"{""depth"": 6, ""weight"": ""sqrt""}","{""iterations"": 800, ""learning_rate"": 0.04, ""l2_leaf_reg"": 10, ""thread_count"": 4}"


**LightGBM: configurations specified before training**

Configuration,Parameters,Fixed settings
1,"{""leaves"": 15, ""weight"": ""none""}","{""n_estimators"": 800, ""learning_rate"": 0.04, ""min_child_samples"": 20, ""reg_lambda"": 5, ""n_jobs"": 4}"
2,"{""leaves"": 15, ""weight"": ""sqrt""}","{""n_estimators"": 800, ""learning_rate"": 0.04, ""min_child_samples"": 20, ""reg_lambda"": 5, ""n_jobs"": 4}"
3,"{""leaves"": 31, ""weight"": ""sqrt""}","{""n_estimators"": 800, ""learning_rate"": 0.04, ""min_child_samples"": 20, ""reg_lambda"": 5, ""n_jobs"": 4}"


In [6]:
def fit_and_predict(family, config_id, seed, scope, fold, partition, save_model=False):
    train_idx, eval_idx = partition["train"], partition["evaluation"]
    y_train = metadata.iloc[train_idx].target.to_numpy(dtype=int)
    model = make_model(family, GRIDS[family][config_id-1], seed, y_train)
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(features.iloc[train_idx], y_train)
    fit_seconds = time.perf_counter() - t0
    t0 = time.perf_counter()
    raw = model.predict_proba(features.iloc[eval_idx])[:,1]
    predict_seconds = time.perf_counter() - t0
    pred = make_predictions(eval_idx, raw, family, seed, fold, scope, config_id)
    resource = {"model":family,"seed":seed,"scope":scope,"fold":fold,"configuration":config_id,
        "training_n":len(train_idx),"evaluation_n":len(eval_idx),"fit_seconds":fit_seconds,
        "batch_predict_seconds":predict_seconds,"fit_warnings":" | ".join(sorted(set(str(w.message) for w in caught)))}
    if save_model:
        checkpoint = MODEL_ROOT / f"{model_slugs[family]}_{scope}_seed{seed}.joblib"
        joblib.dump(model,checkpoint,compress=3)
        resource["model_bytes"] = checkpoint.stat().st_size
        # Warm-up is separate; latency is median of seven fixed-size batches, preprocessing included.
        batch = features.iloc[eval_idx[:min(1000,len(eval_idx))]]
        model.predict_proba(batch)
        durations = []
        for _ in range(7):
            t0 = time.perf_counter()
            model.predict_proba(batch)
            durations.append(time.perf_counter()-t0)
        resource["median_ms_per_1000"] = np.median(durations)*1000*1000/len(batch)
        resource["latency_batch_n"] = len(batch)
    resource_rows.append(resource)
    return pred


def search_family(family, scope):
    for config_id, _ in enumerate(GRIDS[family], start=1):
        for fold, partition in search_splits[scope].items():
            pred = fit_and_predict(family,config_id,42,scope,fold,partition)
            prediction_frames.append(pred)
            search_rows.append({"Model":family,"Scope":scope,"Configuration":config_id,"Fold":fold,
                "AP":average_precision_score(pred.target,pred.raw_score),
                "ROC-AUC":roc_auc_score(pred.target,pred.raw_score),
                "Brier (raw)":brier_score_loss(pred.target,pred.raw_score)})
        # Persist each completed candidate: a later interruption does not erase its evidence.
        pd.concat([p for p in prediction_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]).to_parquet(
            PRED_ROOT / f"search_{model_slugs[family]}_{scope}.parquet",index=False)
        pd.DataFrame(search_rows).to_csv(TABLES / "nb03_search_trials.csv",index=False)
        pd.DataFrame(resource_rows).to_csv(TABLES / "nb03_fit_resources.csv",index=False)
    subset = pd.DataFrame(search_rows).query("Model == @family and Scope == @scope")
    summary = subset.groupby("Configuration",as_index=False).agg(
        **{"Mean AP":("AP","mean"),"Fold SD AP":("AP","std"),"Minimum AP":("AP","min"),
           "Mean ROC-AUC":("ROC-AUC","mean")})
    best = int(summary.sort_values(["Mean AP","Configuration"],ascending=[False,True]).iloc[0].Configuration)
    chosen[(family,scope)] = best
    scientific_table(subset, f"nb03_trials_{model_slugs[family]}_{scope}",
                     f"{family}: all forward-fold evaluations, {scope}")
    scientific_table(summary, f"nb03_selection_{model_slugs[family]}_{scope}",
                     f"{family}: mean across folds; selected configuration {best}, {scope}")
    return best


## 3. Simple reference rules

In [7]:
rule_frames = []
for scope, partition in [("temporal",splits["development"]),("transfer",splits["site_transfer"])]:
    for fold, part in list(search_splits[scope].items()) + [("development",partition)]:
        indices = part["evaluation"]
        view = features.iloc[indices]
        rule_scores = {"Majority":np.full(len(indices),metadata.iloc[part["train"]].target.mean()),
            "Power-drop rule":np.clip(-view.delta_1d.to_numpy()/100,0,1),
            "Recent-level deviation":np.clip(((7*view.power_mean_7d-view.power_pct)/6-view.power_pct).to_numpy()/100,0,1)}
        for family,score in rule_scores.items():
            frame = make_predictions(indices,score,family,42,fold,scope,1)
            frame["decision_fixed"] = (score>=0.2) if family != "Majority" else False
            rule_frames.append(frame)
rule_predictions = pd.concat(rule_frames,ignore_index=True)
rule_predictions.to_parquet(PRED_ROOT / "rules.parquet",index=False)
rule_summary = []
for (family,scope), frame in rule_predictions.query("fold == 'development'").groupby(["model","scope"],sort=False):
    rule_summary.append({"Model":family,"Scope":scope,"AP":average_precision_score(frame.target,frame.raw_score),
        "ROC-AUC":roc_auc_score(frame.target,frame.raw_score),"Precision":precision_score(frame.target,frame.decision_fixed,zero_division=0),
        "Recall":recall_score(frame.target,frame.decision_fixed),"F1":f1_score(frame.target,frame.decision_fixed)})
scientific_table(pd.DataFrame(rule_summary), "nb03_fixed_rules", "Simple rules: development 2023 at the fixed operating point")


**Simple rules: development 2023 at the fixed operating point**

Model,Scope,AP,ROC-AUC,Precision,Recall,F1
Majority,temporal,0.0021,0.5000,0.0000,0.0000,0.0000
Power-drop rule,temporal,0.7690,0.9039,0.8205,0.7619,0.7901
Recent-level deviation,temporal,0.9135,0.9993,0.7059,0.8571,0.7742
Majority,transfer,0.0027,0.5000,0.0000,0.0000,0.0000
Power-drop rule,transfer,0.7948,0.9328,0.8462,0.7333,0.7857
Recent-level deviation,transfer,0.8782,0.9992,0.7692,0.6667,0.7143


## 4. Forward-fold configuration search

### Logistic Regression

In [8]:
selected_config = search_family("Logistic Regression", "temporal")
gc.collect();

**Logistic Regression: mean across folds; selected configuration 3, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8077,0.0880,0.7067,0.9967
2,0.7750,0.0610,0.7147,0.9966
3,0.8119,0.0268,0.7908,0.9963


### Random Forest

In [9]:
selected_config = search_family("Random Forest", "temporal")
gc.collect();

**Random Forest: mean across folds; selected configuration 3, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8111,0.0490,0.7627,0.9731
2,0.8260,0.0261,0.7988,0.9726
3,0.8329,0.0422,0.7844,0.9777


### CatBoost

In [10]:
selected_config = search_family("CatBoost", "temporal")
gc.collect();

**CatBoost: mean across folds; selected configuration 1, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8451,0.0155,0.8291,0.9971
2,0.8199,0.0437,0.7703,0.9969
3,0.8305,0.0306,0.8074,0.9969


### LightGBM

In [11]:
selected_config = search_family("LightGBM", "temporal")
gc.collect();

**LightGBM: mean across folds; selected configuration 2, temporal**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8022,0.0308,0.7667,0.9966
2,0.8155,0.0472,0.7804,0.9974
3,0.8093,0.0550,0.7739,0.9974


## 5. Configuration search for held-site transfer

### Logistic Regression

In [12]:
selected_config = search_family("Logistic Regression", "transfer")
gc.collect();

**Logistic Regression: mean across folds; selected configuration 3, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8247,0.0545,0.7879,0.9980
2,0.7895,0.0609,0.7374,0.9975
3,0.8437,0.0476,0.8018,0.9984


### Random Forest

In [13]:
selected_config = search_family("Random Forest", "transfer")
gc.collect();

**Random Forest: mean across folds; selected configuration 1, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8926,0.0297,0.8694,0.9874
2,0.8900,0.0574,0.8266,0.9872
3,0.8830,0.0440,0.8502,0.9799


### CatBoost

In [14]:
selected_config = search_family("CatBoost", "transfer")
gc.collect();

**CatBoost: mean across folds; selected configuration 2, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8896,0.0481,0.8479,0.9982
2,0.8968,0.0498,0.8455,0.9990
3,0.8952,0.0433,0.8599,0.9991


### LightGBM

In [15]:
selected_config = search_family("LightGBM", "transfer")
gc.collect();

**LightGBM: mean across folds; selected configuration 1, transfer**

Configuration,Mean AP,Fold SD AP,Minimum AP,Mean ROC-AUC
1,0.8948,0.0493,0.8491,0.9989
2,0.8808,0.0419,0.8388,0.9985
3,0.8787,0.0456,0.8318,0.9986


## 6. Repeated training of selected configurations

In [16]:
selected_oof_frames, development_frames = [], []


def repeat_selected(family, scope):
    config_id = chosen[(family,scope)]
    final_part = splits["development" if scope == "temporal" else "site_transfer"]
    for seed in SEEDS:
        if seed == 42:
            existing = [p for p in prediction_frames if p.model.iloc[0] == family
                and p.scope.iloc[0] == scope and p.configuration.iloc[0] == config_id]
            assert len(existing) == len(forward_names)
            selected_oof_frames.extend(existing)
        else:
            for fold, part in search_splits[scope].items():
                selected_oof_frames.append(fit_and_predict(family,config_id,seed,scope,fold,part))
        pred = fit_and_predict(family,config_id,seed,scope,"development",final_part,save_model=True)
        development_frames.append(pred)
        pd.concat([p for p in selected_oof_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]).to_parquet(
            PRED_ROOT / f"oof_{model_slugs[family]}_{scope}.parquet",index=False)
        pd.concat([p for p in development_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]).to_parquet(
            PRED_ROOT / f"development_{model_slugs[family]}_{scope}.parquet",index=False)
        pd.DataFrame(resource_rows).to_csv(TABLES / "nb03_fit_resources.csv",index=False)
    rows = [{"Seed":p.seed.iloc[0],"AP":average_precision_score(p.target,p.raw_score),
        "ROC-AUC":roc_auc_score(p.target,p.raw_score),"Positive days":int(p.target.sum())}
        for p in development_frames if p.model.iloc[0]==family and p.scope.iloc[0]==scope]
    scientific_table(pd.DataFrame(rows), f"nb03_seeds_{model_slugs[family]}_{scope}",
                     f"{family}: development 2023, {scope}; parameters selected using preceding folds only")


### Logistic Regression: temporal

In [17]:
repeat_selected("Logistic Regression", "temporal")
gc.collect();

### Random Forest: temporal

In [18]:
repeat_selected("Random Forest", "temporal")
gc.collect();

### CatBoost: temporal

In [19]:
repeat_selected("CatBoost", "temporal")
gc.collect();

### LightGBM: temporal

In [20]:
repeat_selected("LightGBM", "temporal")
gc.collect();

### Logistic Regression: transfer

In [21]:
repeat_selected("Logistic Regression", "transfer")
gc.collect();

### Random Forest: transfer

In [22]:
repeat_selected("Random Forest", "transfer")
gc.collect();

### CatBoost: transfer

In [23]:
repeat_selected("CatBoost", "transfer")
gc.collect();

### LightGBM: transfer

In [24]:
repeat_selected("LightGBM", "transfer")
gc.collect();

## 7. Calibration and operating points without using 2024–2025

In [25]:
oof = pd.concat(selected_oof_frames + [rule_predictions.query("fold != 'development'")],ignore_index=True)
development = pd.concat(development_frames + [rule_predictions.query("fold == 'development'")],ignore_index=True)
assert not oof.duplicated(["model","scope","seed","observation_id"]).any()
assert not development.duplicated(["model","scope","seed","observation_id"]).any()


def raw_logit(values):
    p = np.clip(np.asarray(values,float),1e-8,1-1e-8)
    return np.log(p/(1-p)).reshape(-1,1)


def fit_sigmoid(frame):
    calibrator = LogisticRegression(C=100,solver="lbfgs",max_iter=1000)
    design = np.zeros((len(frame),1)) if frame.model.iloc[0] == "Majority" else raw_logit(frame.raw_score)
    calibrator.fit(design,frame.target.astype(int))
    assert calibrator.coef_[0,0] >= -1e-10
    return calibrator


def select_thresholds(y, score):
    y,score = np.asarray(y,int),np.asarray(score,float)
    candidates = np.unique(np.r_[score,np.nextafter(score.max(),np.inf)])
    order = np.argsort(-score,kind="stable")
    ordered_y, ordered_s = y[order],score[order]
    cum_tp, cum_fp = np.cumsum(ordered_y),np.cumsum(1-ordered_y)
    end = np.r_[ordered_s[:-1] != ordered_s[1:],True]
    thresholds = np.r_[np.nextafter(score.max(),np.inf),ordered_s[end]]
    tp,fp = np.r_[0,cum_tp[end]],np.r_[0,cum_fp[end]]
    fn = y.sum()-tp
    f1 = np.divide(2*tp,2*tp+fp+fn,out=np.zeros(len(tp),float),where=(2*tp+fp+fn)>0)
    best = np.flatnonzero(np.isclose(f1,f1.max(),rtol=0,atol=1e-14))
    threshold_f1 = float(thresholds[best].max())
    allowed = fp/max((y==0).sum(),1)*1000 <= protocol["false_alarm_limit_per_1000_control_days"]
    best_tp = tp[allowed].max()
    threshold_budget = float(thresholds[allowed & (tp==best_tp)].max())
    return threshold_f1,threshold_budget


calibration_rows, threshold_rows = [], []
for (family,scope,seed), frame in oof.groupby(["model","scope","seed"],sort=False):
    calibrator = fit_sigmoid(frame)
    target = (development.model==family)&(development.scope==scope)&(development.seed==seed)
    subset = development.loc[target]
    development.loc[target,"probability"] = calibrator.predict_proba(raw_logit(subset.raw_score))[:,1]
    t_f1,t_budget = select_thresholds(frame.target,frame.raw_score)
    if family == "Majority":
        t_f1,t_budget = np.inf,np.inf
    development.loc[target,"decision_f1"] = subset.raw_score.to_numpy()>=t_f1
    development.loc[target,"decision_budget"] = subset.raw_score.to_numpy()>=t_budget
    threshold_rows.append({"Model":family,"Scope":scope,"Seed":seed,"OOF F1 threshold":t_f1,
        "OOF alarm-budget threshold":t_budget,"OOF positive days":int(frame.target.sum())})
    joblib.dump({"calibrator":calibrator,"threshold_f1":t_f1,"threshold_budget":t_budget,
        "fit_period":"forward OOF 2020-2022","purpose":"development only"},
        MODEL_ROOT / f"calibration_{model_slugs[family]}_{scope}_seed{seed}.joblib",compress=3)
    early, late = frame.query("fold != 'forward_2022'"), frame.query("fold == 'forward_2022'")
    past_calibrator = fit_sigmoid(early)
    late_probability = past_calibrator.predict_proba(raw_logit(late.raw_score))[:,1]
    calibration_rows.append({"Model":family,"Scope":scope,"Seed":seed,
        "2022 Brier, raw":brier_score_loss(late.target,late.raw_score),
        "2022 Brier, sigmoid":brier_score_loss(late.target,late_probability),
        "2022 log loss, sigmoid":log_loss(late.target,late_probability,labels=[0,1])})
development[["decision_f1","decision_budget"]] = development[["decision_f1","decision_budget"]].astype(bool)
oof.to_parquet(PRED_ROOT / "selected_oof.parquet",index=False)
development.to_parquet(PRED_ROOT / "development_predictions.parquet",index=False)
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(threshold_rows).query("Scope == @scope and Seed == 42").drop(columns=["Scope","Seed"]),
        f"nb03_thresholds_{scope}",f"OOF-derived operating points: {scope}; seed 42")
    scientific_table(pd.DataFrame(calibration_rows).query("Scope == @scope and Seed == 42").drop(columns=["Scope","Seed"]),
        f"nb03_calibration_diagnostic_{scope}",f"Transfer of OOF calibration to 2022: {scope}; seed 42")


**OOF-derived operating points: temporal; seed 42**

Model,OOF F1 threshold,OOF alarm-budget threshold,OOF positive days
Logistic Regression,0.4734,0.0769,117
Random Forest,0.5653,0.1883,117
CatBoost,0.4525,0.0854,117
LightGBM,0.7037,0.0734,117
Majority,∞,∞,117
Power-drop rule,0.0800,0.0800,117
Recent-level deviation,0.5000,0.0400,117


**Transfer of OOF calibration to 2022: temporal; seed 42**

Model,"2022 Brier, raw","2022 Brier, sigmoid","2022 log loss, sigmoid"
Logistic Regression,0.0006,0.0006,0.0027
Random Forest,0.0005,0.0006,0.0039
CatBoost,0.0005,0.0005,0.0024
LightGBM,0.0007,0.0006,0.0029
Majority,0.0018,0.0018,0.0134
Power-drop rule,0.0010,0.0010,0.0071
Recent-level deviation,0.0007,0.0008,0.0046


**OOF-derived operating points: transfer; seed 42**

Model,OOF F1 threshold,OOF alarm-budget threshold,OOF positive days
Logistic Regression,0.6501,0.0714,80
Random Forest,0.5325,0.1475,80
CatBoost,0.8225,0.2575,80
LightGBM,0.0699,0.0699,80
Majority,∞,∞,80
Power-drop rule,0.0800,0.0800,80
Recent-level deviation,0.5000,0.0400,80


**Transfer of OOF calibration to 2022: transfer; seed 42**

Model,"2022 Brier, raw","2022 Brier, sigmoid","2022 log loss, sigmoid"
Logistic Regression,0.0005,0.0005,0.0020
Random Forest,0.0005,0.0005,0.0016
CatBoost,0.0005,0.0004,0.0016
LightGBM,0.0005,0.0005,0.0020
Majority,0.0016,0.0016,0.0122
Power-drop rule,0.0008,0.0009,0.0066
Recent-level deviation,0.0006,0.0006,0.0030


## 8. Comparable development metrics

In [26]:
def evaluate_frame(frame, decision="decision_f1"):
    y = frame.target.to_numpy(dtype=int)
    p = frame.probability.to_numpy(float)
    predicted = frame[decision].to_numpy(dtype=bool)
    tn,fp,fn,tp = confusion_matrix(y,predicted,labels=[0,1]).ravel()
    positive = frame.loc[frame.target.eq(1)]
    assert positive.event_episode_id.notna().all()
    event_recall = positive.groupby("event_episode_id")[decision].max().mean()
    return {"AP":average_precision_score(y,frame.raw_score),"ROC-AUC":roc_auc_score(y,frame.raw_score),
        "Precision":precision_score(y,predicted,zero_division=0),"Recall":recall_score(y,predicted),
        "F1":f1_score(y,predicted),"MCC":matthews_corrcoef(y,predicted),
        "Balanced accuracy":balanced_accuracy_score(y,predicted),"Brier":brier_score_loss(y,p),
        "Log loss":log_loss(y,p,labels=[0,1]),"Event Recall":event_recall,
        "False alarms/1000":fp/max(fp+tn,1)*1000,"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),
        "Positive episodes":positive.event_episode_id.nunique(),"n":len(frame)}


metric_rows = []
for (family,scope,seed),frame in development.groupby(["model","scope","seed"],sort=False):
    metric_rows.append({"Model":family,"Scope":scope,"Seed":seed,**evaluate_frame(frame)})
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(TABLES / "nb03_all_metrics_by_seed.csv",index=False)
for scope in ["temporal","transfer"]:
    summary = metrics.query("Scope == @scope").groupby("Model",sort=False).agg(
        **{"Mean AP":("AP","mean"),"SD AP":("AP","std"),"Minimum AP":("AP","min"),
           "Mean ROC-AUC":("ROC-AUC","mean"),"Repeats":("Seed","size")}).reset_index()
    summary["SD AP"] = summary["SD AP"].where(summary.Repeats.gt(1),"Deterministic rule; one run")
    scientific_table(summary.sort_values("Mean AP",ascending=False),f"nb03_ranking_{scope}",
                     f"Development ranking: {scope}; the mean across seeds is not an ensemble prediction")
    representative = metrics.query("Scope == @scope and Seed == 42")
    scientific_table(representative[["Model","Precision","Recall","F1","MCC","Balanced accuracy"]],
        f"nb03_threshold_metrics_{scope}",f"SCRAM classification at the OOF threshold: {scope}, seed 42")
    scientific_table(representative[["Model","Brier","Log loss","Event Recall","False alarms/1000"]],
        f"nb03_probability_event_metrics_{scope}",f"Calibration and event-level diagnostics: {scope}, seed 42")
    scientific_table(representative[["Model","TN","FP","FN","TP"]],
        f"nb03_confusion_{scope}",f"Confusion counts: {scope}, seed 42")


**Development ranking: temporal; the mean across seeds is not an ensemble prediction**

Model,Mean AP,SD AP,Minimum AP,Mean ROC-AUC,Repeats
LightGBM,0.9480,0.0000,0.9480,0.9980,3
CatBoost,0.9405,0.0116,0.9321,0.9993,3
Logistic Regression,0.9340,0.0000,0.9340,0.9999,3
Recent-level deviation,0.9135,Deterministic rule; one run,0.9135,0.9993,1
Random Forest,0.9045,0.0039,0.9021,0.9878,3
Power-drop rule,0.7690,Deterministic rule; one run,0.7690,0.9039,1
Majority,0.0021,Deterministic rule; one run,0.0021,0.5000,1


**SCRAM classification at the OOF threshold: temporal, seed 42**

Model,Precision,Recall,F1,MCC,Balanced accuracy
Logistic Regression,0.8947,0.8095,0.8500,0.8508,0.9047
Random Forest,0.8864,0.9286,0.9070,0.9070,0.9642
CatBoost,0.9302,0.9524,0.9412,0.9411,0.9761
LightGBM,0.9070,0.9286,0.9176,0.9175,0.9642
Majority,0.0000,0.0000,0.0000,0.0000,0.5000
Power-drop rule,0.8250,0.7857,0.8049,0.8047,0.8927
Recent-level deviation,0.8293,0.8095,0.8193,0.8190,0.9046


**Calibration and event-level diagnostics: temporal, seed 42**

Model,Brier,Log loss,Event Recall,False alarms/1000
Logistic Regression,0.0004,0.0016,0.8095,0.1971
Random Forest,0.0005,0.0022,0.9286,0.2464
CatBoost,0.0003,0.0013,0.9524,0.1478
LightGBM,0.0003,0.0014,0.9286,0.1971
Majority,0.0021,0.0148,0.0000,0.0000
Power-drop rule,0.0008,0.0050,0.7857,0.3450
Recent-level deviation,0.0007,0.0024,0.8095,0.3450


... (6 outputs left out)

**Calibration and event-level diagnostics: transfer, seed 42**

Model,Brier,Log loss,Event Recall,False alarms/1000
Logistic Regression,0.0009,0.0028,0.6667,0.1780
Random Forest,0.0008,0.0031,0.8000,0.3559
CatBoost,0.0006,0.0027,0.8667,0.3559
LightGBM,0.0007,0.0035,0.8667,0.3559
Majority,0.0027,0.0186,0.0000,0.0000
Power-drop rule,0.0010,0.0054,0.8000,0.3559
Recent-level deviation,0.0011,0.0036,0.6000,0.3559


**Confusion counts: transfer, seed 42**

Model,TN,FP,FN,TP
Logistic Regression,5618,1,5,10
Random Forest,5617,2,3,12
CatBoost,5617,2,2,13
LightGBM,5617,2,2,13
Majority,5619,0,15,0
Power-drop rule,5617,2,3,12
Recent-level deviation,5617,2,6,9


In [27]:
budget_rows = []
for (family,scope),frame in development.query("seed == 42").groupby(["model","scope"],sort=False):
    m = evaluate_frame(frame,"decision_budget")
    budget_rows.append({"Model":family,"Scope":scope,**{k:m[k] for k in ["Precision","Recall","F1","Event Recall","False alarms/1000"]}})
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(budget_rows).query("Scope == @scope").drop(columns="Scope"),
        f"nb03_alarm_budget_{scope}",f"OOF alarm-budget threshold: observed performance on {scope}")


**OOF alarm-budget threshold: observed performance on temporal**

Model,Precision,Recall,F1,Event Recall,False alarms/1000
Logistic Regression,0.7692,0.9524,0.8511,0.9524,0.5914
Random Forest,0.8333,0.9524,0.8889,0.9524,0.3943
CatBoost,0.7736,0.9762,0.8632,0.9762,0.5914
LightGBM,0.8200,0.9762,0.8913,0.9762,0.4435
Majority,0.0000,0.0000,0.0000,0.0000,0.0000
Power-drop rule,0.8250,0.7857,0.8049,0.7857,0.3450
Recent-level deviation,0.7321,0.9762,0.8367,0.9762,0.7392


**OOF alarm-budget threshold: observed performance on transfer**

Model,Precision,Recall,F1,Event Recall,False alarms/1000
Logistic Regression,0.8235,0.9333,0.8750,0.9333,0.5339
Random Forest,0.8750,0.9333,0.9032,0.9333,0.3559
CatBoost,0.8235,0.9333,0.8750,0.9333,0.5339
LightGBM,0.8667,0.8667,0.8667,0.8667,0.3559
Majority,0.0000,0.0000,0.0000,0.0000,0.0000
Power-drop rule,0.8571,0.8000,0.8276,0.8000,0.3559
Recent-level deviation,0.8235,0.9333,0.8750,0.9333,0.5339


## 9. Rare events, documented controls and operating regimes

In [28]:
diagnostic = development.query("seed == 42").copy()
row_indices = diagnostic.row_index.to_numpy(int)
diagnostic["power_pct"] = features.iloc[row_indices].power_pct.to_numpy()
diagnostic["delta_1d"] = features.iloc[row_indices].delta_1d.to_numpy()
diagnostic["weather_missing_fraction"] = features.iloc[row_indices].weather_missing_fraction.to_numpy()
diagnostic["change_group"] = pd.cut(np.abs(diagnostic.delta_1d),[-np.inf,2,20,np.inf],
    labels=["Change ≤2 pp","Change >2–20 pp","Change >20 pp"],right=True).astype(str)
diagnostic["weather_group"] = np.where(diagnostic.weather_missing_fraction>0,"At least one weather value missing","Weather complete")
diagnostic["power_group"] = np.where(diagnostic.power_pct.eq(0),"Zero output","Positive output")
hard_rows, subtype_rows, regime_rows = [], [], []
for (family,scope),frame in diagnostic.groupby(["model","scope"],sort=False):
    hard = frame.loc[frame.target.eq(1)|frame.control_kind.eq("Documented non-SCRAM")]
    if hard.target.nunique()==2:
        hm = evaluate_frame(hard)
        hard_rows.append({"Model":family,"Scope":scope,"n":len(hard),"SCRAM prevalence":hard.target.mean(),
            "AP":hm["AP"],"Recall":hm["Recall"],"False alarms/1000":hm["False alarms/1000"]})
    for subtype,part in frame.loc[frame.target.eq(1)].groupby("scram_type",dropna=False):
        subtype_rows.append({"Model":family,"Scope":scope,"SCRAM type":str(subtype),"Positive days":len(part),
            "Detected days":int(part.decision_f1.sum()),"Recall":part.decision_f1.mean()})
    for grouping in ["change_group","power_group","weather_group"]:
        for group,part in frame.groupby(grouping,observed=True):
            positives = part.target.eq(1)
            negatives = part.target.eq(0)
            regime_rows.append({"Model":family,"Scope":scope,"Stratum":group,"Positive days":int(positives.sum()),
                "Negative days":int(negatives.sum()),"Recall":part.loc[positives,"decision_f1"].mean() if positives.any() else "No SCRAM days",
                "False alarms/1000":part.loc[negatives,"decision_f1"].mean()*1000 if negatives.any() else "No control days"})
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(hard_rows).query("Scope == @scope").drop(columns="Scope"),
        f"nb03_hard_controls_{scope}",f"Documented non-SCRAM controls only: {scope}")
    for subtype,part in pd.DataFrame(subtype_rows).query("Scope == @scope").groupby("SCRAM type"):
        scientific_table(part.drop(columns=["Scope","SCRAM type"]),f"nb03_subtype_{subtype}_{scope}",
                         f"Binary detection of rare subtype {subtype}: {scope}")
for (scope,stratum),part in pd.DataFrame(regime_rows).groupby(["Scope","Stratum"],sort=False):
    slug = hashlib.sha256(stratum.encode()).hexdigest()[:8]
    scientific_table(part.drop(columns=["Scope","Stratum"]),f"nb03_regime_{slug}_{scope}",
                     f"{scope}: {stratum}")


**Documented non-SCRAM controls only: temporal**

Model,n,SCRAM prevalence,AP,Recall,False alarms/1000
Logistic Regression,183,0.2295,0.9369,0.8095,28.3688
Random Forest,183,0.2295,0.9088,0.9286,35.4610
CatBoost,183,0.2295,0.9592,0.9524,21.2766
LightGBM,183,0.2295,0.9579,0.9286,28.3688
Majority,183,0.2295,0.2295,0.0000,0.0000
Power-drop rule,183,0.2295,0.8265,0.7857,49.6454
Recent-level deviation,183,0.2295,0.9273,0.8095,49.6454


**Binary detection of rare subtype A: temporal**

Model,Positive days,Detected days,Recall
Logistic Regression,30,26,0.8667
Random Forest,30,28,0.9333
CatBoost,30,30,1.0000
LightGBM,30,29,0.9667
Majority,30,0,0.0000
Power-drop rule,30,24,0.8000
Recent-level deviation,30,25,0.8333


**Binary detection of rare subtype M: temporal**

Model,Positive days,Detected days,Recall
Logistic Regression,12,8,0.6667
Random Forest,12,11,0.9167
CatBoost,12,10,0.8333
LightGBM,12,10,0.8333
Majority,12,0,0.0000
Power-drop rule,12,9,0.7500
Recent-level deviation,12,9,0.7500


... (2 outputs left out)

**Binary detection of rare subtype A: transfer**

Model,Positive days,Detected days,Recall
Logistic Regression,10,8,0.8000
Random Forest,10,9,0.9000
CatBoost,10,9,0.9000
LightGBM,10,10,1.0000
Majority,10,0,0.0000
Power-drop rule,10,9,0.9000
Recent-level deviation,10,6,0.6000


**Binary detection of rare subtype M: transfer**

Model,Positive days,Detected days,Recall
Logistic Regression,5,2,0.4000
Random Forest,5,3,0.6000
CatBoost,5,4,0.8000
LightGBM,5,3,0.6000
Majority,5,0,0.0000
Power-drop rule,5,3,0.6000
Recent-level deviation,5,3,0.6000


## 10. Complementary errors and site diagnostics

In [29]:
pair_rows = []
for scope in ["temporal","transfer"]:
    frame = diagnostic.query("scope == @scope and model != 'Majority'")
    truth = frame.groupby("observation_id").target.first()
    decisions = frame.pivot(index="observation_id",columns="model",values="decision_f1").loc[truth.index]
    assert decisions.notna().all().all()
    for first,second in itertools.combinations(decisions.columns,2):
        pos = truth.eq(1)
        a,b = decisions.loc[pos,first].astype(bool),decisions.loc[pos,second].astype(bool)
        union_miss = ((~a)|(~b)).sum()
        pair_rows.append({"Scope":scope,"Model A":first,"Model B":second,
            "Only A detects":int((a&~b).sum()),"Only B detects":int((~a&b).sum()),
            "Both miss":int((~a&~b).sum()),
            "Miss Jaccard":float((~a&~b).sum()/union_miss) if union_miss else 0.0})
for scope in ["temporal","transfer"]:
    pair_table = pd.DataFrame(pair_rows).query("Scope == @scope").drop(columns="Scope")
    for block,offset in enumerate(range(0,len(pair_table),5),start=1):
        scientific_table(pair_table.iloc[offset:offset+5],f"nb03_error_overlap_{scope}_{block}",
                         f"Complementarity of missed events: {scope}")


**Complementarity of missed events: temporal**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
CatBoost,LightGBM,2,1,1,0.2500
CatBoost,Logistic Regression,6,0,2,0.2500
CatBoost,Power-drop rule,8,1,1,0.1000
CatBoost,Random Forest,2,1,1,0.2500
CatBoost,Recent-level deviation,7,1,1,0.1111


**Complementarity of missed events: temporal**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
LightGBM,Logistic Regression,5,0,3,0.3750
LightGBM,Power-drop rule,6,0,3,0.3333
LightGBM,Random Forest,1,1,2,0.5000
LightGBM,Recent-level deviation,5,0,3,0.3750
Logistic Regression,Power-drop rule,5,4,4,0.3077


**Complementarity of missed events: temporal**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
Logistic Regression,Random Forest,1,6,2,0.2222
Logistic Regression,Recent-level deviation,3,3,5,0.4545
Power-drop rule,Random Forest,1,7,2,0.2000
Power-drop rule,Recent-level deviation,5,6,3,0.2143
Random Forest,Recent-level deviation,6,1,2,0.2222


... (2 outputs left out)

**Complementarity of missed events: transfer**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
LightGBM,Logistic Regression,3,0,2,0.4000
LightGBM,Power-drop rule,1,0,2,0.6667
LightGBM,Random Forest,1,0,2,0.6667
LightGBM,Recent-level deviation,4,0,2,0.3333
Logistic Regression,Power-drop rule,0,2,3,0.6000


**Complementarity of missed events: transfer**

Model A,Model B,Only A detects,Only B detects,Both miss,Miss Jaccard
Logistic Regression,Random Forest,0,2,3,0.6000
Logistic Regression,Recent-level deviation,2,1,4,0.5714
Power-drop rule,Random Forest,0,0,3,1.0000
Power-drop rule,Recent-level deviation,3,0,3,0.5000
Random Forest,Recent-level deviation,3,0,3,0.5000


In [30]:
# Seed 42 and model ordering are fixed; selection of the displayed leader uses development only.
leader = metrics.query("Scope == 'temporal' and Model in @FAMILIES").groupby("Model").AP.mean().idxmax()
leader_frame = diagnostic.query("scope == 'temporal' and model == @leader")
site_rows, reactor_rows = [], []
for grouping,container in [("plant_id",site_rows),("reactor_id",reactor_rows)]:
    for name,part in leader_frame.groupby(grouping,sort=True):
        positive,negative = part.target.eq(1),part.target.eq(0)
        container.append({"Site" if grouping=="plant_id" else "Reactor":name,
            "Positive days":int(positive.sum()),"Negative days":int(negative.sum()),
            "TP":int((positive&part.decision_f1).sum()),"FN":int((positive&~part.decision_f1).sum()),
            "FP":int((negative&part.decision_f1).sum()),
            "Recall":part.loc[positive,"decision_f1"].mean() if positive.any() else "No SCRAM days"})
for label,rows in [("site",site_rows),("reactor",reactor_rows)]:
    table = pd.DataFrame(rows)
    table.to_csv(TABLES / f"nb03_all_{label}_errors.csv",index=False)
    # Retain every site/reactor with an event or a false alarm in the CSV ledgers; the narrative keeps missed-event evidence.
    informative = table.loc[table["Positive days"].gt(0)|table.FP.gt(0)].copy()
    for block,offset in enumerate(range(0,len(informative),8),start=1):
        scientific_table(informative.iloc[offset:offset+8],f"nb03_{label}_errors_{block}",
                         f"{leader}: {label}, events or false alarms; all eligible groups")
all_errors = diagnostic.loc[diagnostic.target.ne(diagnostic.decision_f1.astype(int))].copy()
all_errors.to_parquet(PRED_ROOT / "development_error_ledger.parquet",index=False)
misses = diagnostic.query("scope == 'temporal' and model != 'Majority' and target == 1 and decision_f1 == False")
missed_events = misses.groupby(["observation_id","date","reactor_id","event_id"],dropna=False).agg(
    **{"Missed by":("model",lambda s:", ".join(sorted(set(s)))),"Power (%)":("power_pct","first"),
       "Change (pp)":("delta_1d","first"),"Weather missing fraction":("weather_missing_fraction","first")}).reset_index().drop(columns="observation_id")
for block,offset in enumerate(range(0,len(missed_events),6),start=1):
    scientific_table(missed_events.iloc[offset:offset+6].fillna({"event_id":"Not specified by source"}),f"nb03_missed_events_{block}",
                     f"All positive days missed by at least one substantive comparator")


**All positive days missed by at least one substantive comparator**

date,reactor_id,event_id,Missed by,Power (%),Change (pp),Weather missing fraction
2023-05-21 00:00:00,05000259,56531,Power-drop rule,0.0000,0.0000,0.0000
2023-09-28 00:00:00,05000263,56759,Random Forest,0.0000,−75.0000,0.2000
2023-11-17 00:00:00,05000318,56856,Power-drop rule,2.0000,2.0000,0.0000
2023-04-21 00:00:00,05000325,56478,Power-drop rule,0.0000,0.0000,0.2000
2023-11-15 00:00:00,05000364,56852,"CatBoost, LightGBM, Logistic Regression, Power-drop rule, Random Forest, Recent-level deviation",0.0000,−1.0000,0.0000
2023-04-01 00:00:00,05000366,56446,"CatBoost, Logistic Regression",0.0000,−71.0000,0.4000


**All positive days missed by at least one substantive comparator**

date,reactor_id,event_id,Missed by,Power (%),Change (pp),Weather missing fraction
2023-11-11 00:00:00,05000387,56846,Power-drop rule,0.0000,0.0000,0.0000
2023-05-31 00:00:00,05000423,56544,Logistic Regression,0.0000,−100.0000,1.0000
2023-08-11 00:00:00,05000440,56673,"Logistic Regression, Power-drop rule",0.0000,0.0000,0.2000
2023-11-18 00:00:00,05000458,56863,"Logistic Regression, Recent-level deviation",0.0000,−21.0000,0.0000
2023-12-14 00:00:00,05000458,56887,Power-drop rule,0.0000,0.0000,0.0000
2023-06-03 00:00:00,05000529,56551,Recent-level deviation,0.0000,−100.0000,0.4000


**All positive days missed by at least one substantive comparator**

date,reactor_id,event_id,Missed by,Power (%),Change (pp),Weather missing fraction
2023-03-16 00:00:00,05200025,56414,"Logistic Regression, Recent-level deviation",0.0000,−18.0000,0.0000
2023-04-11 00:00:00,05200025,56460,"LightGBM, Logistic Regression, Power-drop rule, Random Forest, Recent-level deviation",0.0000,0.0000,0.0000
2023-05-03 00:00:00,05200025,56497,"LightGBM, Logistic Regression, Power-drop rule, Recent-level deviation",0.0000,0.0000,0.0000
2023-07-10 00:00:00,05200025,56614,Recent-level deviation,0.0000,−60.0000,0.0000
2023-07-22 00:00:00,05200025,56632,Recent-level deviation,0.0000,−51.0000,0.0000


## 11. Numerical PR profiles and probability reliability

In [31]:
pr_rows = []
for (family,scope),part in diagnostic.groupby(["model","scope"],sort=False):
    precision,recall,threshold = precision_recall_curve(part.target,part.raw_score)
    row = {"Model":family,"Scope":scope}
    for goal in [0.5,0.75,0.9,1.0]:
        row[f"Precision at recall ≥{goal:.2f}"] = float(precision[recall>=goal-1e-12].max())
    pr_rows.append(row)
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(pr_rows).query("Scope == @scope").drop(columns="Scope"),
        f"nb03_pr_profile_{scope}",f"Descriptive development PR points: {scope}")
reliability = leader_frame[["probability","target"]].copy()
reliability["bin"] = pd.qcut(reliability.probability,10,duplicates="drop")
reliability_table = reliability.groupby("bin",observed=True).agg(
    n=("target","size"),positive_days=("target","sum"),mean_probability=("probability","mean"),
    observed_frequency=("target","mean"),minimum_probability=("probability","min"),maximum_probability=("probability","max")).reset_index(drop=True)
reliability_table.insert(0,"Bin",np.arange(1,len(reliability_table)+1))
reliability_table["positive_days"] = reliability_table.positive_days.astype(int)
scientific_table(reliability_table,"nb03_reliability_leader",f"Probability reliability: {leader}, seed 42")


**Descriptive development PR points: temporal**

Model,Precision at recall ≥0.50,Precision at recall ≥0.75,Precision at recall ≥0.90,Precision at recall ≥1.00
Logistic Regression,0.9545,0.9143,0.8696,0.7119
Random Forest,0.9412,0.9412,0.8889,0.0021
CatBoost,0.9655,0.9412,0.9318,0.4468
LightGBM,0.9714,0.9714,0.9500,0.0245
Majority,0.0021,0.0021,0.0021,0.0021
Power-drop rule,1.0000,0.8421,0.0021,0.0021
Recent-level deviation,1.0000,0.9189,0.7321,0.0873


**Descriptive development PR points: transfer**

Model,Precision at recall ≥0.50,Precision at recall ≥0.75,Precision at recall ≥0.90,Precision at recall ≥1.00
Logistic Regression,0.9167,0.8667,0.8333,0.8333
Random Forest,0.9000,0.8750,0.8750,0.1327
CatBoost,0.9231,0.9231,0.8750,0.2174
LightGBM,0.9231,0.9231,0.8235,0.0584
Majority,0.0027,0.0027,0.0027,0.0027
Power-drop rule,1.0000,0.8571,0.0027,0.0027
Recent-level deviation,1.0000,0.8235,0.8235,0.2174


**Probability reliability: LightGBM, seed 42**

Bin,n,positive_days,mean_probability,observed_frequency,minimum_probability,maximum_probability
1,2034,0,1.31e−05,0.0000,4.41e−06,1.70e−05
2,2034,0,2.01e−05,0.0000,1.70e−05,2.32e−05
3,2032,0,2.62e−05,0.0000,2.32e−05,2.92e−05
4,2033,0,3.23e−05,0.0000,2.92e−05,3.52e−05
5,2034,0,3.86e−05,0.0000,3.52e−05,4.20e−05
6,2033,0,4.60e−05,0.0000,4.20e−05,5.04e−05
7,2033,0,5.57e−05,0.0000,5.04e−05,6.15e−05
8,2033,0,6.93e−05,0.0000,6.15e−05,7.88e−05
9,2033,0,9.51e−05,0.0000,7.88e−05,0.0001
10,2034,42,0.0179,0.0206,0.0001,0.9951


## 12. Computational efficiency

In [32]:
resource_frame = pd.DataFrame(resource_rows)
assert resource_frame.fit_warnings.fillna("").eq("").all(), "Inspect recorded fit warnings before accepting model results"
for scope in ["temporal","transfer"]:
    final_resources = resource_frame.query("scope == @scope and fold == 'development'").groupby("model",sort=False).agg(
        **{"Mean fit (s)":("fit_seconds","mean"),"SD fit (s)":("fit_seconds","std"),
           "Median batch ms/1000":("median_ms_per_1000","median"),"Mean checkpoint (MiB)":("model_bytes",lambda s:s.mean()/2**20)}).reset_index()
    scientific_table(final_resources,f"nb03_efficiency_{scope}",f"Development models: {scope}; measured resource use")
effort = resource_frame.groupby(["model","scope"],sort=False).agg(
    fits=("fit_seconds","size"),total_fit_seconds=("fit_seconds","sum")).reset_index()
scientific_table(effort,"nb03_total_search_effort","Fit budget for search and seed repetitions, including unselected configurations")
selected_configurations = [{"model":family,"scope":scope,"configuration":config_id,
    "parameters":GRIDS[family][config_id-1],"fixed":BASE_SETTINGS[family]}
    for (family,scope),config_id in chosen.items()]
(OUTPUT_ROOT / "Configs/selected_baselines.json").write_text(json.dumps(selected_configurations,indent=2));


**Development models: temporal; measured resource use**

model,Mean fit (s),SD fit (s),Median batch ms/1000,Mean checkpoint (MiB)
Logistic Regression,0.5565,0.0224,0.7618,0.0034
Random Forest,26.7287,0.4740,42.2854,0.5361
CatBoost,7.5949,0.1165,0.8883,0.1093
LightGBM,4.9496,0.0957,8.0391,0.5168


**Development models: transfer; measured resource use**

model,Mean fit (s),SD fit (s),Median batch ms/1000,Mean checkpoint (MiB)
Logistic Regression,0.3624,0.0102,0.7737,0.0034
Random Forest,9.5878,0.0682,41.8256,0.6357
CatBoost,5.4418,0.2653,0.7915,0.1045
LightGBM,3.8006,0.3430,7.1029,0.4921


**Fit budget for search and seed repetitions, including unselected configurations**

model,scope,fits,total_fit_seconds
Logistic Regression,temporal,18,7.8488
Random Forest,temporal,18,225.9970
CatBoost,temporal,18,89.1184
LightGBM,temporal,18,61.0912
Logistic Regression,transfer,18,4.6121
Random Forest,transfer,18,107.6415
CatBoost,transfer,18,62.6210
LightGBM,transfer,18,46.3009


## 13. Training-history sensitivity

In [33]:
learning_history_rows=[]
for last_year in [2019,2020,2022]:
    tr=splits["development"]["train"]
    tr=tr[metadata.iloc[tr].date.dt.year.le(last_year).to_numpy()]
    if last_year==2022:
        prediction=development.query("model == @leader and scope == 'temporal' and seed == 42")
    else:
        prediction=fit_and_predict(leader,chosen[(leader,"temporal")],42,"temporal",f"history_to_{last_year}",{"train":tr,"evaluation":splits["development"]["evaluation"]})
    learning_history_rows.append({"Model":leader,"Last training year":last_year,"Training n":len(tr),"Training SCRAM days":int(metadata.iloc[tr].target.sum()),"Development AP":average_precision_score(prediction.target,prediction.raw_score)})
scientific_table(pd.DataFrame(learning_history_rows),"nb03_history_learning_curve","Training-history extent on the common 2023 evaluation cohort")


**Training-history extent on the common 2023 evaluation cohort**

Model,Last training year,Training n,Training SCRAM days,Development AP
LightGBM,2019,28287,49,0.8713
LightGBM,2020,57744,96,0.9538
LightGBM,2022,116904,170,0.9480


## 14. Implications for modern comparators and RACF-Net

In [34]:
handoff_rows=[]
for family in metrics.Model.unique():
    temporal=metrics.query("Model == @family and Scope == 'temporal'")
    transfer=metrics.query("Model == @family and Scope == 'transfer'")
    hard=pd.DataFrame(hard_rows).query("Model == @family and Scope == 'temporal'")
    strongest="Compact deterministic reference" if family not in FAMILIES else "Learned combination of common predictors"
    if family=="Logistic Regression": strongest="Regularized linear reference; transparent additive score"
    if family in ["CatBoost","LightGBM","Random Forest"]: strongest="Nonlinear interactions without a deep temporal encoder"
    limitation="Performance depends on the selected cohort and threshold transfer"
    if family=="Majority":limitation="No event discrimination; prevalence reference only"
    if family in ["Power-drop rule","Recent-level deviation"]:limitation="Fixed rule cannot learn distinct causes or site-specific exceptions"
    handoff_rows.append({"Model":family,"Strength":strongest,"Limitation to test":limitation})
scientific_table(pd.DataFrame(handoff_rows),"nb03_design_handoff","Baseline evidence motivating RACF-Net component tests")
(OUTPUT_ROOT / "Configs/baseline_design_evidence.json").write_text(json.dumps({"leader":leader,"ranking_temporal":metrics.query("Scope == 'temporal'").groupby("Model").AP.mean().to_dict(),"ranking_transfer":metrics.query("Scope == 'transfer'").groupby("Model").AP.mean().to_dict(),"design_trace_ids":["D1","D2","D3","D4","D5","D6","D7"],"next":"Compare modern temporal models before selecting RACF-Net components","test_used":False},indent=2));


**Baseline evidence motivating RACF-Net component tests**

Model,Strength,Limitation to test
Logistic Regression,Regularized linear reference; transparent additive score,Performance depends on the selected cohort and threshold transfer
Random Forest,Nonlinear interactions without a deep temporal encoder,Performance depends on the selected cohort and threshold transfer
CatBoost,Nonlinear interactions without a deep temporal encoder,Performance depends on the selected cohort and threshold transfer
LightGBM,Nonlinear interactions without a deep temporal encoder,Performance depends on the selected cohort and threshold transfer
Majority,Compact deterministic reference,No event discrimination; prevalence reference only
Power-drop rule,Compact deterministic reference,Fixed rule cannot learn distinct causes or site-specific exceptions
Recent-level deviation,Compact deterministic reference,Fixed rule cannot learn distinct causes or site-specific exceptions


## 15. Checkpoint reproducibility and model complexity

In [35]:
reload_rows=[]
for family in FAMILIES:
    for scope in ["temporal","transfer"]:
        checkpoint=MODEL_ROOT / f"{model_slugs[family]}_{scope}_seed42.joblib"
        model=joblib.load(checkpoint)
        reference=development.query("model == @family and scope == @scope and seed == 42")
        raw=model.predict_proba(features.iloc[reference.row_index.to_numpy(int)])[:,1]
        difference=float(np.max(np.abs(raw-reference.raw_score.to_numpy())))
        assert difference<=1e-10
        if family=="Logistic Regression":
            complexity=int(model[-1].coef_.size+model[-1].intercept_.size);unit="Linear coefficients"
        elif family=="Random Forest":
            complexity=int(sum(t.tree_.node_count for t in model[-1].estimators_));unit="Tree nodes"
        elif family=="CatBoost":
            complexity=int(np.sum(model.get_tree_leaf_counts()));unit="Tree leaves"
        else:
            complexity=int(sum(t["num_leaves"] for t in model.booster_.dump_model()["tree_info"]));unit="Tree leaves"
        reload_rows.append({"Model":family,"Scope":scope,"n":len(raw),"Maximum score difference":difference,"Complexity":complexity,"Unit":unit})
        del model;gc.collect()
for scope in ["temporal","transfer"]:
    scientific_table(pd.DataFrame(reload_rows).query("Scope == @scope").drop(columns="Scope"),"nb03_checkpoint_reproduction_"+scope,f"Checkpoint reload agreement and model complexity: {scope}")


**Checkpoint reload agreement and model complexity: temporal**

Model,n,Maximum score difference,Complexity,Unit
Logistic Regression,20333,0.0000,66,Linear coefficients
Random Forest,20333,0.0000,27694,Tree nodes
CatBoost,20333,0.0000,12800,Tree leaves
LightGBM,20333,0.0000,11997,Tree leaves


**Checkpoint reload agreement and model complexity: transfer**

Model,n,Maximum score difference,Complexity,Unit
Logistic Regression,5634,0.0000,66,Linear coefficients
Random Forest,5634,0.0000,26724,Tree nodes
CatBoost,5634,0.0000,12800,Tree leaves
LightGBM,5634,0.0000,11914,Tree leaves
